<a href="https://colab.research.google.com/github/abhsrivastava/hugging_face_transformers/blob/main/HuggingFace_Spaces_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# In this demo, we will deploy an application to hugging face spaces

In [1]:
# Install dependencies

%pip install -q --upgrade "huggingface_hub<=2.5.0" gradio_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 89.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires gradio-client==2.5.0, but you have gradio-client 2.6.0 which is incompatible.


In [17]:
# Import packages

from huggingface_hub import HfApi, login, get_token
import os
import warnings
import time

warnings.filterwarnings('ignore')
print(f'✅ Setup complete')


✅ Setup complete


In [15]:
# Login into HuggingFace

login()
api = HfApi()
user_info = api.whoami()
USERNAME=user_info['name']
HF_TOKEN = get_token()
print(f'Currently Logging in as: {USERNAME}')

Currently Logging in as: abhishes


In [8]:
# Now that login is done. let us create the huggingface space

SPACE_NAME = "novapay-chatbot"
REPO_ID = f"{USERNAME}/{SPACE_NAME}"
api.create_repo(
    repo_id = REPO_ID,
    repo_type = "space",
    space_sdk="gradio",
    exist_ok = True
    )

print(f"✅ Space created: https://huggingface.co/spaces/{REPO_ID}")

✅ Space created: https://huggingface.co/spaces/abhishes/novapay-chatbot


In [31]:
app_code = r"""
import os
import warnings

import gradio as gr
from huggingface_hub import InferenceClient

warnings.filterwarnings("ignore")

print(f"Gradio Version: {gr.__version__}")
print("✅ Imports successful")


MODELS_TO_TRY = [
    "abhishes/novapay-sentiment",
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "meta-llama/Llama-3.3-70B-Instruct",
]

client = None
ACTIVE_MODEL = None

for model_id in MODELS_TO_TRY:
    try:
        test_client = InferenceClient(model=model_id)

        test_client.chat_completion(
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=5,
        )

        client = test_client
        ACTIVE_MODEL = model_id

        print(f"✅ Successfully connected to {model_id}")
        break

    except Exception as ex:
        print(
            f"⚠️ {model_id} unavailable: "
            f"{type(ex).__name__}: {ex}"
        )

if client is None:
    print("\n❌ No models available. Check your HF token or try again later.")
else:
    print(f"💡 Active model: {ACTIVE_MODEL}")
    print("💡 This model runs on HF servers—no local GPU needed!")


def extract_text(content):
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts = []

        for block in content:
            if isinstance(block, dict):
                if block.get("type") == "text":
                    text_parts.append(block.get("text", ""))
            else:
                text_parts.append(str(block))

        return "\n".join(text_parts)

    return str(content)


def respond(message, history):
    if client is None:
        yield "No Hugging Face inference model is currently available."
        return

    messages = [
        {
            "role": "system",
            "content": (
                "You are NovaPay's customer support assistant. "
                "Be helpful, concise, and professional."
            ),
        }
    ]

    for history_message in history:
        role = history_message.get("role")
        content = extract_text(history_message.get("content", ""))

        if role in {"user", "assistant"} and content:
            messages.append(
                {
                    "role": role,
                    "content": content,
                }
            )

    messages.append(
        {
            "role": "user",
            "content": message,
        }
    )

    partial = ""

    try:
        for chunk in client.chat_completion(
            messages=messages,
            max_tokens=256,
            stream=True,
        ):
            token = chunk.choices[0].delta.content or ""
            partial += token
            yield partial

    except Exception as ex:
        yield f"Model request failed: {type(ex).__name__}: {ex}"


demo = gr.ChatInterface(
    fn=respond,
    title="NovaPay Support ChatBot v2",
    description=(
        "Now with streaming! Tokens appear progressively. "
        "Same model, dramatically better UX."
    ),
    api_name="chat",
)

demo.launch()
"""

print("=== app.py content ===")
print(app_code)

# Verify the generated Python before uploading it
compile(app_code, "app.py", "exec")
print("✅ Generated app.py passes Python syntax validation")

=== app.py content ===

import os
import warnings

import gradio as gr
from huggingface_hub import InferenceClient

warnings.filterwarnings("ignore")

print(f"Gradio Version: {gr.__version__}")
print("✅ Imports successful")


MODELS_TO_TRY = [
    "abhishes/novapay-sentiment",
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "meta-llama/Llama-3.3-70B-Instruct",
]

client = None
ACTIVE_MODEL = None

for model_id in MODELS_TO_TRY:
    try:
        test_client = InferenceClient(model=model_id)

        test_client.chat_completion(
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=5,
        )

        client = test_client
        ACTIVE_MODEL = model_id

        print(f"✅ Successfully connected to {model_id}")
        break

    except Exception as ex:
        print(
            f"⚠️ {model_id} unavailable: "
            f"{type(ex).__name__}: {ex}"
        )

if client is None:
    print("

In [32]:
# Upload the code into spaces

api.upload_file(
    path_or_fileobj=app_code.encode(),
    path_in_repo="app.py",
    repo_id=REPO_ID,
    repo_type="space",
)

print("✅ app.py uploaded to Space!")

✅ app.py uploaded to Space!


In [33]:
# Let us also create a requirements.txt file

requirements = "huggingface_hub\n"

api.upload_file(
    path_or_fileobj=requirements.encode(),
    path_in_repo="requirements.txt",
    repo_id=REPO_ID,
    repo_type="space",
)

print("✅ requirements.txt uploaded!")

No files have been modified since last commit. Skipping to prevent empty commit.


✅ requirements.txt uploaded!


In [34]:
#add HF token to space

api.add_space_secret(
    repo_id=REPO_ID,
    key="HF_TOKEN",
    value=HF_TOKEN,
)

print("✅ HF_TOKEN secret added to Space!")
print("💡 The token is stored securely — it is NOT visible in the source code.")


✅ HF_TOKEN secret added to Space!
💡 The token is stored securely — it is NOT visible in the source code.


In [35]:
# Now let us verify the deployment


SPACE_URL = f"https://huggingface.co/spaces/{REPO_ID}"
EMBED_URL = f"https://{USERNAME}-{SPACE_NAME}.hf.space"

print(f"🌐 Space URL: {SPACE_URL}")
print(f"🔗 Embed URL: {EMBED_URL}")
print(f"\n⏳ Polling for Space to reach RUNNING state (up to 3 minutes)...")

for attempt in range(18):  # 18 × 10s = 3 minutes
    runtime = api.get_space_runtime(repo_id=REPO_ID)
    stage = runtime.stage
    print(f"   [{attempt*10:>3}s] stage = {stage}")
    if stage == "RUNNING":
        print(f"\n✅ Space is RUNNING!")
        break
    if stage in ("BUILD_ERROR", "RUNTIME_ERROR", "CONFIG_ERROR"):
        print(f"\n❌ Space failed to start: {stage}")
        print(f"   Check logs at {SPACE_URL}")
        break
    time.sleep(10)
else:
    print(f"\n⚠️  Space did not reach RUNNING in 3 minutes. Check logs at {SPACE_URL}")

🌐 Space URL: https://huggingface.co/spaces/abhishes/novapay-chatbot
🔗 Embed URL: https://abhishes-novapay-chatbot.hf.space

⏳ Polling for Space to reach RUNNING state (up to 3 minutes)...
   [  0s] stage = RUNNING_BUILDING
   [ 10s] stage = RUNNING_BUILDING
   [ 20s] stage = RUNNING_BUILDING
   [ 30s] stage = RUNNING_BUILDING
   [ 40s] stage = RUNNING_BUILDING
   [ 50s] stage = RUNNING_BUILDING
   [ 60s] stage = RUNNING_BUILDING
   [ 70s] stage = RUNNING_APP_STARTING
   [ 80s] stage = RUNNING_APP_STARTING
   [ 90s] stage = RUNNING

✅ Space is RUNNING!


In [36]:
# Now that deployment is done let's connect to the space programmatically


# Connect to the deployed Space using the Gradio client
from gradio_client import Client
import time

print("Connecting to deployed Space...")
gradio_client = None
for attempt in range(6):  # 6 × 10s = 1 minute of retries
    try:
        gradio_client = Client(REPO_ID, token=HF_TOKEN)
        print("✅ Connected to deployed Space!")
        break
    except Exception as e:
        print(f"   Attempt {attempt+1}/6 — {type(e).__name__}: {e}")
        time.sleep(10)

if gradio_client is None:
    print(f"\n⚠️  Could not connect after 1 minute. Check {SPACE_URL}")

Connecting to deployed Space...
Loaded as API: https://abhishes-novapay-chatbot.hf.space
✅ Connected to deployed Space!


In [37]:
# Let's run some inference

# Batch-process 5 test messages through the deployed chatbot
test_messages = [
    "What are NovaPay business hours?",
    "I need to dispute a charge on my account.",
    "How do I set up automatic bill payments?",
    "My card was lost, what should I do?",
    "Can I send money internationally with NovaPay?",
]

DEFAULT_SYSTEM = "You are NovaPay customer support assistant. Help with payments, transfers, and account issues."
DEFAULT_TONE = 0.5

print("=== Batch Query Results ===")
for i, msg in enumerate(test_messages, 1):
    try:
        # ChatInterface predict signature: (message, system_prompt, tone) → str
        # api_name "/chat" matches the explicit api_name set in app.py
        result = gradio_client.predict(
            msg,
            DEFAULT_SYSTEM,
            DEFAULT_TONE,
            api_name="/chat",
        )
        result_text = str(result)
        snippet = result_text[:200] + "..." if len(result_text) > 200 else result_text
        print(f"\n📨 Message {i}: {msg}")
        print(f"🤖 Response: {snippet}")
    except Exception as e:
        print(f"\n📨 Message {i}: {msg}")
        print(f"❌ Error: {type(e).__name__}: {e}")

=== Batch Query Results ===

📨 Message 1: What are NovaPay business hours?
🤖 Response: NovaPay's business hours are Monday through Friday, 9:00 AM to 5:00 PM EST. Our support team is available to assist you during these hours. If you have a urgent matter outside of business hours, you c...

📨 Message 2: I need to dispute a charge on my account.
🤖 Response: I'd be happy to help you with that. To initiate a dispute, please provide me with the following information:

* Your account name and number
* The date and amount of the disputed transaction
* A clear...

📨 Message 3: How do I set up automatic bill payments?
🤖 Response: Setting up automatic bill payments is a convenient way to ensure timely payments. To do so, please follow these steps:

1. Log in to your NovaPay account online or through the mobile app.
2. Navigate ...

📨 Message 4: My card was lost, what should I do?
🤖 Response: If your card has been lost or stolen, please contact us immediately so we can assist you in securing you